In [ ]:
# 1. 필요한 라이브러리 설치
!pip install pmaw pandas matplotlib seaborn wordcloud vaderSentiment openpyxl plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
from collections import Counter
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 감정 분석
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Reddit 데이터 수집
from pmaw import PushshiftAPI
import datetime as dt

# 시각화
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ 모든 라이브러리 설치 및 import 완료!")


In [ ]:
# 2. Reddit 키워드 감정 분석기 클래스
class RedditSentimentAnalyzer:
    def __init__(self):
        self.api = PushshiftAPI()
        self.sentiment_analyzer = SentimentIntensityAnalyzer()
        self.data = None
        self.results = None
        
    def collect_reddit_data(self, keywords, subreddits=None, days_back=30, limit_per_keyword=200):
        """Reddit 데이터 수집"""
        print("🚀 Reddit 데이터 수집 시작...")
        print(f"🔍 검색 키워드: {', '.join(keywords)}")
        print(f"📅 수집 기간: 최근 {days_back}일")
        
        # 날짜 설정
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        
        all_posts = []
        
        # 각 키워드별로 데이터 수집
        for keyword in keywords:
            print(f"\n🔎 '{keyword}' 키워드 검색 중...")
            
            if subreddits:
                # 특정 서브레딧에서 검색
                for subreddit in subreddits:
                    try:
                        posts = self.api.search_submissions(
                            subreddit=subreddit,
                            q=keyword,
                            limit=limit_per_keyword // len(subreddits),
                            after=int(start_date.timestamp()),
                            before=int(end_date.timestamp())
                        )
                        
                        posts_list = list(posts)
                        for post in posts_list:
                            post['search_keyword'] = keyword
                        
                        all_posts.extend(posts_list)
                        print(f"   📊 r/{subreddit}: {len(posts_list)}개")
                        
                    except Exception as e:
                        print(f"   ❌ r/{subreddit} 오류: {str(e)[:50]}...")
            else:
                # 전체 Reddit 검색
                try:
                    posts = self.api.search_submissions(
                        q=keyword,
                        limit=limit_per_keyword,
                        after=int(start_date.timestamp()),
                        before=int(end_date.timestamp())
                    )
                    
                    posts_list = list(posts)
                    for post in posts_list:
                        post['search_keyword'] = keyword
                    
                    all_posts.extend(posts_list)
                    print(f"   📊 전체 Reddit: {len(posts_list)}개")
                    
                except Exception as e:
                    print(f"   ❌ '{keyword}' 검색 오류: {str(e)[:50]}...")
        
        if all_posts:
            self.data = pd.DataFrame(all_posts)
            # 중복 제거
            self.data = self.data.drop_duplicates(subset=['id'])
            print(f"\n🎉 총 {len(self.data)}개 게시물 수집 완료!")
            return True
        else:
            print("\n❌ 수집된 데이터가 없습니다.")
            return False
    
    def preprocess_and_analyze(self):
        """텍스트 전처리 및 감정 분석"""
        if self.data is None:
            print("❌ 데이터가 없습니다.")
            return False
        
        print("\n🔧 텍스트 전처리 중...")
        
        # 텍스트 전처리
        def clean_text(text):
            if pd.isna(text):
                return ""
            # URL 제거
            text = re.sub(r'http\S+|www\S+|https\S+', '', text)
            # 특수문자 정리
            text = re.sub(r'[^\w\s\.\!\?\,\-]', ' ', text)
            # 연속 공백 제거
            text = re.sub(r'\s+', ' ', text)
            return text.strip()
        
        self.data['title_clean'] = self.data['title'].fillna('').apply(clean_text)
        self.data['selftext_clean'] = self.data['selftext'].fillna('').apply(clean_text)
        self.data['combined_text'] = self.data['title_clean'] + ' ' + self.data['selftext_clean']
        
        print("😊 감정 분석 수행 중...")
        
        # 감정 분석
        sentiment_results = []
        for text in self.data['combined_text']:
            if len(text.strip()) > 5:
                scores = self.sentiment_analyzer.polarity_scores(text)
                sentiment_results.append(scores)
            else:
                sentiment_results.append({'compound': 0, 'pos': 0, 'neu': 1, 'neg': 0})
        
        # 감정 점수 추가
        self.data['sentiment_compound'] = [s['compound'] for s in sentiment_results]
        self.data['sentiment_positive'] = [s['pos'] for s in sentiment_results]
        self.data['sentiment_negative'] = [s['neg'] for s in sentiment_results]
        self.data['sentiment_neutral'] = [s['neu'] for s in sentiment_results]
        
        # 감정 라벨
        def get_sentiment_label(score):
            if score >= 0.05:
                return 'Positive'
            elif score <= -0.05:
                return 'Negative'
            else:
                return 'Neutral'
        
        self.data['sentiment_label'] = self.data['sentiment_compound'].apply(get_sentiment_label)
        
        # 날짜 변환
        self.data['created_date'] = pd.to_datetime(self.data['created_utc'], unit='s')
        
        print("✅ 전처리 및 감정 분석 완료!")
        
        # 결과 요약
        print(f"\n📊 감정 분석 결과 요약:")
        sentiment_summary = self.data['sentiment_label'].value_counts()
        for sentiment, count in sentiment_summary.items():
            percentage = (count / len(self.data)) * 100
            print(f"   {sentiment}: {count}개 ({percentage:.1f}%)")
        
        return True
    
    def analyze_by_keywords(self):
        """키워드별 상세 분석"""
        if self.data is None:
            return None
        
        print("\n🔍 키워드별 분석 중...")
        
        keyword_analysis = []
        
        for keyword in self.data['search_keyword'].unique():
            keyword_data = self.data[self.data['search_keyword'] == keyword]
            
            analysis = {
                'keyword': keyword,
                'total_posts': len(keyword_data),
                'avg_score': keyword_data['score'].mean() if 'score' in keyword_data.columns else 0,
                'avg_sentiment_score': keyword_data['sentiment_compound'].mean(),
                'positive_count': len(keyword_data[keyword_data['sentiment_label'] == 'Positive']),
                'negative_count': len(keyword_data[keyword_data['sentiment_label'] == 'Negative']),
                'neutral_count': len(keyword_data[keyword_data['sentiment_label'] == 'Neutral']),
                'positive_ratio': len(keyword_data[keyword_data['sentiment_label'] == 'Positive']) / len(keyword_data) * 100,
                'negative_ratio': len(keyword_data[keyword_data['sentiment_label'] == 'Negative']) / len(keyword_data) * 100
            }
            
            keyword_analysis.append(analysis)
        
        self.results = pd.DataFrame(keyword_analysis)
        
        print("✅ 키워드별 분석 완료!")
        return self.results
    
    def create_visualizations(self):
        """시각화 생성"""
        if self.data is None:
            return
        
        print("\n📊 시각화 생성 중...")
        
        # 1. 전체 감정 분포
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 감정 분포 파이차트
        sentiment_counts = self.data['sentiment_label'].value_counts()
        axes[0,0].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%')
        axes[0,0].set_title('전체 감정 분포')
        
        # 감정 점수 히스토그램
        axes[0,1].hist(self.data['sentiment_compound'], bins=30, alpha=0.7)
        axes[0,1].set_title('감정 점수 분포')
        axes[0,1].set_xlabel('Sentiment Score')
        axes[0,1].set_ylabel('Frequency')
        
        # 키워드별 감정 분포
        if 'search_keyword' in self.data.columns:
            keyword_sentiment = self.data.groupby(['search_keyword', 'sentiment_label']).size().unstack(fill_value=0)
            keyword_sentiment.plot(kind='bar', stacked=True, ax=axes[1,0])
            axes[1,0].set_title('키워드별 감정 분포')
            axes[1,0].tick_params(axis='x', rotation=45)
        
        # 시간별 감정 트렌드
        daily_sentiment = self.data.groupby([self.data['created_date'].dt.date, 'sentiment_label']).size().unstack(fill_value=0)
        daily_sentiment.plot(ax=axes[1,1])
        axes[1,1].set_title('일별 감정 트렌드')
        axes[1,1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # 2. 워드클라우드
        self.create_wordclouds()
    
    def create_wordclouds(self):
        """워드클라우드 생성"""
        print("☁️ 워드클라우드 생성 중...")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 전체 텍스트 워드클라우드
        all_text = ' '.join(self.data['combined_text'].dropna())
        if all_text.strip():
            wordcloud = WordCloud(width=400, height=300, background_color='white', 
                                max_words=100).generate(all_text)
            axes[0,0].imshow(wordcloud, interpolation='bilinear')
            axes[0,0].axis('off')
            axes[0,0].set_title('전체 키워드')
        
        # 감정별 워드클라우드
        sentiments = ['Positive', 'Negative', 'Neutral']
        positions = [(0,1), (1,0), (1,1)]
        
        for sentiment, pos in zip(sentiments, positions):
            sentiment_text = ' '.join(
                self.data[self.data['sentiment_label'] == sentiment]['combined_text'].dropna()
            )
            
            if sentiment_text.strip():
                wordcloud = WordCloud(width=400, height=300, background_color='white',
                                    max_words=50).generate(sentiment_text)
                axes[pos].imshow(wordcloud, interpolation='bilinear')
                axes[pos].axis('off')
                axes[pos].set_title(f'{sentiment} 리뷰 키워드')
            else:
                axes[pos].text(0.5, 0.5, f'No {sentiment}\ndata available', 
                             ha='center', va='center', fontsize=12)
                axes[pos].set_xlim(0, 1)
                axes[pos].set_ylim(0, 1)
                axes[pos].axis('off')
                axes[pos].set_title(f'{sentiment} 리뷰 키워드')
        
        plt.tight_layout()
        plt.show()
    
    def save_to_excel(self, filename='reddit_sentiment_analysis'):
        """결과를 엑셀 파일로 저장"""
        if self.data is None:
            print("❌ 저장할 데이터가 없습니다.")
            return
        
        print(f"\n💾 엑셀 파일로 저장 중: {filename}.xlsx")
        
        with pd.ExcelWriter(f'{filename}.xlsx', engine='openpyxl') as writer:
            # 1. 전체 데이터
            save_columns = [
                'title', 'selftext', 'score', 'num_comments', 'subreddit',
                'search_keyword', 'created_date', 'sentiment_label', 
                'sentiment_compound', 'sentiment_positive', 'sentiment_negative'
            ]
            
            available_columns = [col for col in save_columns if col in self.data.columns]
            self.data[available_columns].to_excel(writer, sheet_name='전체_데이터', index=False)
            
            # 2. 키워드별 요약
            if self.results is not None:
                self.results.to_excel(writer, sheet_name='키워드별_요약', index=False)
            
            # 3. 감정별 통계
            sentiment_stats = self.data.groupby('sentiment_label').agg({
                'sentiment_compound': ['mean', 'std', 'count'],
                'score': ['mean', 'std'] if 'score' in self.data.columns else ['count']
            }).round(3)
            
            sentiment_stats.to_excel(writer, sheet_name='감정별_통계')
            
            # 4. 일별 통계
            daily_stats = self.data.groupby([
                self.data['created_date'].dt.date, 'sentiment_label'
            ]).size().unstack